# A12. Sunrise Problem (일출 문제)

**핵심 질문**: 지구의 나이가 45억 4천만년이라면, 내일 해가 뜰 확률은?

---

## 학습 목표
1. 라플라스의 계승 규칙(Rule of Succession) 이해
2. Beta 분포를 활용한 베이지안 확률 갱신 이해
3. 귀납법의 한계와 Black Swan 개념 학습

## 1. 환경 설정

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# 한글 폰트 설정 (Malgun Gothic)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# Seaborn 스타일
sns.set_theme(style='whitegrid', font='Malgun Gothic')

# 고해상도 출력
plt.rcParams['figure.dpi'] = 120

print('환경 설정 완료')

## 2. Sunrise Problem 역사적 맥락

### Pierre-Simon Laplace (1749-1827)

프랑스 수학자 **라플라스**는 1814년 저서 *"Essai philosophique sur les probabilités"* 에서 다음과 같은 질문을 던졌습니다:

> *"해가 매일 떠오르는 것을 관찰해왔다. 내일도 해가 뜰 확률은 얼마인가?"*

이 질문은 단순해 보이지만, **귀납적 추론(inductive reasoning)**의 본질을 건드립니다.

### 라플라스의 계승 규칙 (Rule of Succession)

라플라스는 이 문제에 대해 다음과 같은 공식을 제시했습니다:

$$P(\text{내일 해가 뜸} \mid n\text{일 연속 관찰}) = \frac{n + 1}{n + 2}$$

여기서:
- $n$ = 연속으로 해가 뜬 것을 관찰한 일수
- **사전 분포(Prior)**: $\text{Beta}(1, 1)$ = 균일 분포 (아무 사전 지식 없음)
- **사후 분포(Posterior)**: $\text{Beta}(n+1, 1)$

### 유도 과정

해가 뜰 확률을 $\theta$라 하면:

1. **사전 분포**: $\theta \sim \text{Beta}(1, 1)$ (균일 분포)
2. **우도**: $n$번 연속 성공 → $\theta^n$
3. **사후 분포**: $\theta \mid \text{data} \sim \text{Beta}(1 + n, 1 + 0) = \text{Beta}(n+1, 1)$
4. **예측 확률**: $E[\theta \mid \text{data}] = \frac{n+1}{n+2}$

## 3. 핵심 계산

In [ ]:
# === 지구 나이 기반 계산 ===

# 지구 나이 (년)
earth_age_years = 45.4e9  # 45억 4천만년

# 1년 = 365.25일 (윤년 포함 평균)
days_per_year = 365.25

# 총 일수 n
n = earth_age_years * days_per_year

print(f"지구 나이: {earth_age_years:.1e} 년")
print(f"1년 평균 일수: {days_per_year} 일")
print(f"총 관찰 일수 (n): {n:.4e} 일")
print(f"                  ≈ {n/1e12:.2f} × 10¹² 일")
print()

# 라플라스의 계승 규칙 적용
p_sunrise = (n + 1) / (n + 2)

# 해가 뜨지 않을 확률
p_no_sunrise = 1 - p_sunrise

print("=" * 60)
print("라플라스의 계승 규칙 (Rule of Succession)")
print("=" * 60)
print(f"P(내일 해가 뜸) = (n+1)/(n+2)")
print(f"               = ({n:.4e} + 1) / ({n:.4e} + 2)")
print(f"               ≈ {p_sunrise}")
print()
print(f"P(내일 해가 안 뜸) = 1/(n+2)")
print(f"                  ≈ {p_no_sunrise:.4e}")
print(f"                  ≈ 6 × 10⁻¹³")
print()
print("해석: 1에 매우 가깝지만, 수학적으로 절대 1이 될 수 없다!")

In [ ]:
# === 관찰 일수별 확률 정리 ===

observations = [
    (1, "첫째 날"),
    (7, "1주일"),
    (30, "1개월"),
    (365, "1년"),
    (3652, "10년"),
    (36525, "100년"),
    (365250, "1,000년"),
    (3.6525e6, "10,000년"),
    (3.6525e9, "100억 일"),
    (n, "지구 나이")
]

print(f"{'관찰 일수 n':>20} {'설명':>12} {'P(해가 뜸)':>25} {'P(안 뜸)':>15}")
print("-" * 75)

for n_obs, desc in observations:
    p = (n_obs + 1) / (n_obs + 2)
    p_not = 1 - p
    print(f"{n_obs:>20.2e} {desc:>12} {p:>25.15f} {p_not:>15.4e}")

### 해석

관찰 일수가 증가할수록:
- $P(\text{해가 뜸})$은 1에 점점 가까워짐
- 그러나 **수학적으로 절대 1이 되지 않음**
- 지구 나이만큼 관찰해도 $P(\text{해가 안 뜸}) \approx 6 \times 10^{-13}$
- 이는 약 **1.66조 분의 1** 확률

## 4. 시각화 1: n에 따른 P 변화 그래프

In [ ]:
# === n에 따른 확률 변화 그래프 ===

# 로그 스케일로 n 값 생성
n_values = np.logspace(0, 12, 500)  # 1 ~ 10^12
p_values = (n_values + 1) / (n_values + 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 왼쪽: P(해가 뜸) ---
ax1 = axes[0]
ax1.semilogx(n_values, p_values, color='#e74c3c', linewidth=2)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='P = 1 (절대 도달 불가)')

# 주요 포인트 표시
key_points = [1, 365, 365250, n]
key_labels = ['n=1', 'n=365\n(1년)', 'n=365,250\n(1000년)', f'n≈1.66×10¹²\n(지구 나이)']
for kn, kl in zip(key_points, key_labels):
    kp = (kn + 1) / (kn + 2)
    ax1.plot(kn, kp, 'o', color='#2c3e50', markersize=8, zorder=5)
    offset_y = -0.08 if kn == 1 else 0.02
    ax1.annotate(kl, (kn, kp), textcoords='offset points',
                 xytext=(10, -25), fontsize=8, ha='left',
                 arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax1.set_xlabel('관찰 일수 n (로그 스케일)', fontsize=11)
ax1.set_ylabel('P(내일 해가 뜸)', fontsize=11)
ax1.set_title('라플라스 계승 규칙: n에 따른 확률 변화', fontsize=12, fontweight='bold')
ax1.set_ylim(0.4, 1.02)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- 오른쪽: P(해가 안 뜸) = 1/(n+2) ---
ax2 = axes[1]
p_not_values = 1 / (n_values + 2)
ax2.loglog(n_values, p_not_values, color='#3498db', linewidth=2)

# 지구 나이 표시
earth_p_not = 1 / (n + 2)
ax2.plot(n, earth_p_not, 'o', color='#e74c3c', markersize=10, zorder=5)
ax2.annotate(f'지구 나이\n≈ 6×10⁻¹³',
             (n, earth_p_not), textcoords='offset points',
             xytext=(-80, 30), fontsize=9,
             arrowprops=dict(arrowstyle='->', color='red', lw=1.2),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax2.set_xlabel('관찰 일수 n (로그 스케일)', fontsize=11)
ax2.set_ylabel('P(내일 해가 안 뜸) (로그 스케일)', fontsize=11)
ax2.set_title('해가 뜨지 않을 확률의 감소', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 해석

- **왼쪽 그래프**: $P(\text{해가 뜸})$은 관찰 횟수가 증가할수록 1에 수렴하지만, **절대 1에 도달하지 않음**
- **오른쪽 그래프**: $P(\text{해가 안 뜸}) = \frac{1}{n+2}$는 $n$에 반비례하여 감소 (로그-로그 스케일에서 직선)
- 지구 나이만큼 관찰해도 "내일 해가 뜬다"를 **100% 확신할 수 없다**는 것이 수학적 결론

## 5. 시각화 2: Beta 분포 관점

In [ ]:
# === Beta 분포 시각화 ===
# Prior: Beta(1,1) → Posterior: Beta(n+1, 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 다양한 n 값에 대한 Beta 분포
cases = [
    (0, "사전 분포: Beta(1,1)\nn = 0 (관찰 전)"),
    (5, "Beta(6,1)\nn = 5 (5일 관찰)"),
    (50, "Beta(51,1)\nn = 50 (50일 관찰)"),
    (500, "Beta(501,1)\nn = 500 (500일 관찰)")
]

colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']

for idx, ((n_obs, title), color) in enumerate(zip(cases, colors)):
    ax = axes[idx // 2][idx % 2]
    alpha_param = n_obs + 1
    beta_param = 1
    
    x = np.linspace(0, 1, 1000)
    y = stats.beta.pdf(x, alpha_param, beta_param)
    
    ax.fill_between(x, y, alpha=0.3, color=color)
    ax.plot(x, y, color=color, linewidth=2)
    
    # 평균(기대값) 표시
    mean_val = alpha_param / (alpha_param + beta_param)
    ax.axvline(x=mean_val, color='black', linestyle='--', linewidth=1.5,
               label=f'E[θ] = {mean_val:.4f}')
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('θ (해가 뜰 확률)', fontsize=10)
    ax.set_ylabel('확률 밀도', fontsize=10)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.suptitle('Beta 분포의 사후 갱신: 관찰 횟수가 증가하면 θ→1 집중',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === 극단적 사후분포: 지구 나이만큼 관찰 시 ===

fig, ax = plt.subplots(figsize=(10, 5))

# 다양한 관찰 횟수의 Beta 분포를 하나의 그래프에
n_list = [1, 10, 100, 1000]
colors_grad = ['#a8d8ea', '#5bb5e0', '#2980b9', '#1a5276']

x = np.linspace(0, 1, 5000)

for n_obs, color in zip(n_list, colors_grad):
    alpha_p = n_obs + 1
    y = stats.beta.pdf(x, alpha_p, 1)
    ax.plot(x, y, color=color, linewidth=2, label=f'n = {n_obs:,} → E[θ] = {alpha_p/(alpha_p+1):.4f}')
    ax.fill_between(x, y, alpha=0.1, color=color)

ax.set_xlabel('θ (해가 뜰 확률)', fontsize=12)
ax.set_ylabel('확률 밀도 f(θ)', fontsize=12)
ax.set_title('관찰 횟수 증가에 따른 사후 분포 Beta(n+1, 1)의 변화',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0.5, 1.01)
ax.grid(True, alpha=0.3)

# 화살표로 수렴 방향 표시
ax.annotate('n → ∞ 일수록\nθ = 1에 집중',
            xy=(0.998, 800), fontsize=11,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
plt.show()

print("\n지구 나이(n ≈ 1.66×10¹²)에서의 Beta(n+1, 1) 분포:")
print("→ θ = 1 근처에 극도로 집중된 분포")
print("→ 그래프로 표현 불가능할 정도로 좁은 폭")
print(f"→ 표준편차 ≈ {1/np.sqrt((n+1)*(n+3)):.4e}")

### 해석

**Beta 분포 관점**에서 Sunrise Problem을 이해하면:

| 단계 | 분포 | 의미 |
|------|------|------|
| 사전 분포 (관찰 전) | $\text{Beta}(1, 1)$ | 해가 뜰 확률 $\theta$에 대해 아무 정보 없음 (균일 분포) |
| $n$일 관찰 후 | $\text{Beta}(n+1, 1)$ | $\theta$가 1에 매우 가깝다는 강한 믿음 |
| 기대값 | $E[\theta] = \frac{n+1}{n+2}$ | 관찰이 쌓일수록 1에 수렴 |
| 분산 | $\text{Var}[\theta] = \frac{(n+1)}{(n+2)^2(n+3)}$ | 불확실성이 급격히 감소 |

**핵심**: 아무리 많은 관찰을 해도 $\theta = 1$을 **확실하게** 주장할 수 없다. 이것이 베이지안 확률의 아름다운 겸손함.

## 6. 철학적 토론

In [ ]:
# === 철학적 토론을 위한 데이터 시각화 ===

fig, ax = plt.subplots(figsize=(12, 6))

# 칠면조 문제 (Bertrand Russell의 비유)
# 1001일 동안 먹이를 받다가 추수감사절에 도축됨
days = np.arange(1, 1002)
confidence = (days) / (days + 1)  # Rule of Succession

# 먹이를 받는 기간
ax.plot(days[:1000], confidence[:1000], color='#27ae60', linewidth=2.5,
        label='칠면조의 확신: "내일도 먹이를 받을 확률"')

# 추수감사절!
ax.plot(1001, 0, 'X', color='#e74c3c', markersize=20, zorder=5,
        label='Day 1001: 추수감사절 (도축!)')

# 급락 표시
ax.plot([1000, 1001], [confidence[999], 0], color='#e74c3c',
        linewidth=3, linestyle='--')

ax.annotate('Black Swan Event!\n(검은 백조 사건)',
            xy=(1001, 0), xytext=(750, 0.3),
            fontsize=13, fontweight='bold', color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2),
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#fadbd8', alpha=0.9))

ax.set_xlabel('날짜', fontsize=12)
ax.set_ylabel('확신 수준 P(내일도 먹이 받음)', fontsize=12)
ax.set_title('버트런드 러셀의 칠면조: 귀납법의 한계', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 해석

#### 6.1 귀납법의 한계

**"내일 해가 뜬다"를 100% 확신할 수 있는가?**

- 귀납적 추론은 과거 관찰에서 미래를 예측하는 것
- 아무리 많은 관찰을 축적해도 **논리적 필연성**은 확보되지 않음
- 이것이 데이비드 흄(David Hume)이 제기한 **귀납의 문제(Problem of Induction)**

#### 6.2 검은 백조 (Black Swan)

나심 탈레브(Nassim Taleb)의 개념:

| 특성 | 설명 |
|------|------|
| **희귀성** | 정상적인 기대 범위를 벗어남 |
| **극단적 충격** | 발생 시 엄청난 영향 |
| **사후 예측 가능성** | 발생 후에는 "예측 가능했다"고 설명 |

- 유럽인들은 모든 백조가 하얗다고 확신했지만, 호주에서 **검은 백조**를 발견
- 관찰된 적 없는 사건의 확률을 0으로 놓으면 위험!

#### 6.3 ML에서의 교훈

머신러닝에서도 동일한 문제가 발생합니다:

- **학습 데이터에 없는 상황**: 모델이 본 적 없는 입력에 대해 어떻게 예측하는가?
- **과적합(Overfitting)**: 훈련 데이터에 대한 확신이 너무 높으면 일반화 실패
- **예측 불확실성(Uncertainty Estimation)**: 모델이 "모른다"고 말할 수 있어야 함
- **라플라스 스무딩(Laplace Smoothing)**: NLP에서 관찰되지 않은 단어의 확률을 0이 아닌 작은 값으로 설정
  - 이것이 바로 Sunrise Problem의 계승 규칙에서 유래!

In [ ]:
# === 최종 요약 ===

print("=" * 65)
print(" A12. Sunrise Problem 최종 요약")
print("=" * 65)
print()
print("[문제]")
print("  지구 나이 = 45.4억 년 동안 매일 해가 떴다면,")
print("  내일 해가 뜰 확률은?")
print()
print("[풀이] 라플라스의 계승 규칙")
print(f"  n = 45.4 × 10⁹ × 365.25 ≈ 1.66 × 10¹² 일")
print(f"  P(내일 해가 뜸) = (n+1)/(n+2)")
print(f"                  ≈ 1 - 6×10⁻¹³")
print(f"                  ≈ 0.999999999999...")
print()
print("[핵심 통찰]")
print("  1. 확률은 1에 수렴하지만 절대 1이 되지 않는다")
print("  2. 귀납적 추론은 논리적 확실성을 보장하지 않는다")
print("  3. 관찰되지 않은 사건(Black Swan)에 대한 겸손이 필요하다")
print("  4. ML의 라플라스 스무딩은 이 원리의 실용적 적용이다")
print("=" * 65)